In [1]:
import json
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "llmr_updated_arch").exists():
    repo_root = repo_root.parent

for rel_path in (
    "llmr_updated_arch/src",
    "krrood/src",
    "pycram/src",
    "semantic_digital_twin/src",
    "uniworld/src",
):
    src_path = repo_root / rel_path
    if src_path.exists() and str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))

from dotenv import load_dotenv
from krrood.entity_query_language.query.match import Match
from pycram.datastructures.grasp import GraspDescription
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from semantic_digital_twin.world_description.world_entity import Body
from uniworld import load_pr2_apartment_world

from llmr_updated_arch import LLMBackend
from llmr_updated_arch.generation.flanagan import FlanaganReasoner
from llmr_updated_arch.generation.framenet import FrameNetReasoner
from llmr_updated_arch.hypotheses import BuildOrchestrator
from llmr_updated_arch.integrations.langchain.provider import LLMProvider, make_llm
from pycram.motion_executor import simulated_robot
from pycram.plans.factories import sequential
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.robot_body import ParkArmsAction, MoveTorsoAction

## World And Model Setup

In [2]:
for env_path in (
    repo_root / "llmr" / ".env",
    repo_root / "llmr_updated_arch" / ".env",
    repo_root / ".env",
):
    if env_path.exists():
        load_dotenv(env_path, override=False)

world, robot_view, context = load_pr2_apartment_world()
context.evaluate_conditions = False

symbol_type = Body

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
print("LLM ready:", getattr(llm, "model_name", llm))

Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


LLM ready: gpt-4o-mini


In [3]:
# ROS setup
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [4]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

ROS2 publishers started


## Backend And Match Helpers

In [ ]:
from pycram.datastructures.enums import Arms
from pycram.locations.locations import CostmapLocation
from pycram.robot_plans.actions.core.robot_body import ParkArmsAction, MoveTorsoAction
from semantic_digital_twin.datastructures.definitions import TorsoState
def prepare_robot_for_pickup(context):
    setup_plan = sequential(
        [
            ParkArmsAction(Arms.BOTH),
            MoveTorsoAction(TorsoState.HIGH),
        ],
        context=context,
    )
    with simulated_robot:
        setup_plan.perform()


def build_navigate_then_pick(action_instance, context):
    pickup_pose = CostmapLocation(
        target=action_instance.object_designator.global_pose,
        reachable=True,
        reachable_arm=action_instance.arm,
        grasp_description=action_instance.grasp_description,
        context=context,
    ).ground()

    return sequential(
        [
            NavigateAction(pickup_pose, keep_joint_states=True),
            PickUpAction(
                object_designator=action_instance.object_designator,
                arm=action_instance.arm,
                grasp_description=action_instance.grasp_description,
            ),
        ],
        context=context,
    )

def fresh_pickup_match():
    return Match(PickUpAction)(
        object_designator=...,
        arm=...,
        grasp_description=Match(GraspDescription)(
            approach_direction=...,
            vertical_alignment=...,
            manipulator=...,
        ),
    )

def role_rows(roles):
    return [
        {
            "display_id": role.display_id,
            "role": role.role_name,
            "family": role.role_family,
            "text": role.filler_text,
            "kind": role.filler_kind,
            "status": role.meta.status.value,
            "grounding": role.meta.grounding.value,
            "run_id": role.meta.short_run_id,
        }
        for role in roles
    ]


def phase_rows(phases):
    return [
        {
            "display_id": phase.display_id,
            "index": phase.phase_index,
            "phase": phase.phase_name,
            "target": phase.target_object,
            "contact": phase.contact,
            "motion_type": phase.motion_type,
            "urgency": phase.urgency,
            "run_id": phase.meta.short_run_id,
        }
        for phase in phases
    ]

def print_pickup_action(action_instance):
    print("Action type:", type(action_instance).__name__)
    print("Object     :", action_instance.object_designator)
    print("Arm        :", action_instance.arm)
    print("Grasp      :", action_instance.grasp_description)


orchestrator = BuildOrchestrator.with_default_builders()
graph = orchestrator.graph


def run_instruction(instruction: str):
    backend = LLMBackend(
        llm=llm,
        symbol_type=symbol_type,
        instruction=instruction,
        strict_required=True,
        reasoners=[
            FrameNetReasoner(llm=llm),
            FlanaganReasoner(llm=llm),
        ],
        sg_model_orchestrator=orchestrator,
    )
    action = next(iter(backend.evaluate(fresh_pickup_match())))
    return action, backend

## Resolve One Instruction

In [ ]:
INSTRUCTION = "pick up the milk from the table"

action, backend = run_instruction(INSTRUCTION)
print_pickup_action(action)

## Execution

In [ ]:
prepare_robot_for_pickup(context)

role1_plan_node = build_navigate_then_pick(action, context)
print("Role 1 Navigate + PickUp plan ready:", role1_plan_node)
with simulated_robot:
    try:
        role1_plan_node.perform()
    except MotionDidNotFinish:
        print("Role 1 motion did not finish; pickup parameters and plan construction were still exercised.")


## Inspect Semantic Outputs

In [ ]:
backend.semantics.slot_filling.model_dump()

In [ ]:
semantics = backend.semantics.model_dump(mode="json")
print(json.dumps(semantics, indent=2, default=repr))

## Inspect Hypothesis Projection

In [ ]:
print("Projection result:", backend.last_build_result)
print("FrameNet nodes:", len(graph.nodes_from_reasoner("framenet_reasoner")))
print("Flanagan nodes:", len(graph.nodes_from_reasoner("flanagan_reasoner")))
print("Total hypothesis nodes:", len(list(graph)))

In [ ]:
from graphviz import Source
from IPython.display import display
from llmr_updated_arch.hypotheses.adapters import to_dot, render_graph

print(f"Hypothesis graph nodes: {len(list(graph))}")
dot = to_dot(graph)
print(f"DOT lines: {len(dot.splitlines())}")

try:
    src = Source(dot, format="svg")
    display(src)
    output_path = repo_root / "llmr_updated_arch" / "notebooks" / "hypo_graph"
    src.render(str(output_path), cleanup=True)
except Exception as exc:
    print(f"Graphviz display failed: {type(exc).__name__}: {exc}")
    print(dot)


In [ ]:
type(action)

In [ ]:
action.object_designator

In [ ]:
backend.semantics.__dict__.keys()

In [ ]:
backend.semantics.__dict__['motion_phases'].model_dump()